In [ ]:
import pandas as pd
import json
import pycld2 as cld2


# cld2

In [ ]:
def langComp_cld2(artist):

    # reading lyrics from file 
    file = open(f'raw_data/lyrics/{artist}_lyrics.txt', 'r', errors='ignore')

    song = None
    songs = []
    song_index = -1

    # iterating through the file 
    for line in file: 
        line = line.strip()

        # getting the song name 
        if line.startswith("SONG NAME ["):
            song_index += 1
            songs.append({})
            song = line[len("SONG NAME ["):-1]
            songs[song_index]['title'] = song
            songs[song_index]['lyrics'] = ''

        # getting the lyrics 
        elif not 'ContributorsTranslations' in line and not 'Read More' in line:
            songs[song_index]['lyrics']+=line

    # iterating through all the songs to get language composition
    for index, song in enumerate(songs):
        songs[index]['lang_comp'] = {}

        # checking if the lyrics are not empty
        if not len(song['lyrics']) == 0:
            # detecting the language composition 
                # (0, 32, 'Korean', 'ko')
                # (bytesOffSet, bytesLength, languageName, languageCode) 
            isReliable, textBytesFound, details, vectors = cld2.detect(song['lyrics'], returnVectors= True)
            for vector in vectors:
                if not vector[2] in songs[index]['lang_comp']:
                    songs[index]['lang_comp'][vector[2].upper()] = vector[1]
                else:
                    songs[index]['lang_comp'][vector[2].upper()] += vector[1]
        else:
            # if the lyrics are empty --> setting the language composition to NONE
            songs[index]['lang_comp'] = {'NONE':0}

        # sorting the language composition by value (AI assisted code)
        songs[index]['lang_comp_sorted'] = dict(sorted(songs[index]['lang_comp'].items(),key=lambda x:x[1], reverse=True))
        del songs[index]['lang_comp']
        del songs[index]['lyrics']

        # calculating the percentage of each language in the song
        bytes = songs[index]['lang_comp_sorted'].values()
        sum_bytes = 0 
        for b in bytes:
            sum_bytes += b
        songs[index]['lang_percent'] = {}
        for lang,value in songs[index]['lang_comp_sorted'].items():
            if sum_bytes != 0:
                songs[index]['lang_percent'][lang.upper()] = round(value/sum_bytes,3)
            else: 
                songs[index]['lang_percent'][lang.upper()] = 0

    fileDump = open(f'{artist}_percent_cld2.json','w')
    json.dump(songs, fileDump, indent=4)


In [ ]:
# files then moved to: processed_data/cld2/lang_percent
artists = ['bts','blackpink','exo','twice']
for artist in artists:
    langComp_cld2(artist)

# Billboard + cld2

In [188]:
# removing non-ascii/english characters 
def englishify(text):
    new_text = ''
    for letter in text:
        if letter.isascii() and (letter != '(' and letter != ')'):
            new_text += letter
    return new_text 

In [ ]:
# getting the language composition percentage of the songs in the Billboard Global chart
def billboardLangPercent(artist):
    # opening the billboard data file
    billboard_file = open(f'raw_data/billboard_data/{artist}_billboard.json', 'r')
    billboard_data = json.load(billboard_file)

    # opening the cld2 (language composition) data file
    cld2_file = open(f'processed_data/cld2/lang_percent/{artist}_percent_cld2.json','r')
    cld2_data = json.load(cld2_file)

    # iterating through the billboard and cld2 data to find matching songs
    for index_b, song in enumerate(billboard_data['billboard_data']):
        song_name_billboard = englishify(song['song_name']).strip().title()
        # print(song['song_name'])
        match = False
        for index_c, song in enumerate(cld2_data):
            song_name_cld2 = englishify(song['title']).strip().title()
            if song_name_billboard == song_name_cld2:
                # print(f'Match: \n{song_name_billboard}\n{song_name_cld2}\n')
                match = True
                billboard_data['billboard_data'][index_b]['lang_percent'] = cld2_data[index_c]['lang_percent']
                break
        if not match:
            billboard_data['billboard_data'][index_b]['lang_percent'] = {'UNKNOWN':1}
            # print(f'No Match: {song_name_billboard}\n')


    fileDump = open(f'{artist}_billboard_lang_percent.json','w')
    json.dump(billboard_data,fileDump,indent=4)


In [ ]:
# files then moved to: processed_data/billboard_lang_percent 
artists = ['bts','blackpink','exo','twice']
for artist in artists:
    billboardLangPercent(artist)

# Spotify + cld2

In [ ]:
# getting the language composition of spotify top tracks 
def tracksLangPercent(artist):

    # getting albums data
    spotify_file = open(f'raw_data/artist_data/{artist}_data.json', 'r')
    spotify_data = json.load(spotify_file)

    # getting lytics language composition data 
    cld2_file = open(f'processed_data/cld2/lang_percent/{artist}_percent_cld2.json','r')
    cld2_data = json.load(cld2_file)

    # iterating through the top_tracks and identify corresponding lyrics composition 
    for index_s, track in enumerate(spotify_data['top_tracks']):
        match = False
        track = englishify(track['name']).strip()
        for index_c, song in enumerate(cld2_data):
            song = englishify(song['title']).strip()
            if track == song:
                # print(f'Match: \n{track}\n{song}\n')
                spotify_data['top_tracks'][index_s]['lang_percent'] = cld2_data[index_c]['lang_percent']
                match = True
                break

        if not match:
            # print(f'No Match: {track}\n')
            spotify_data['top_tracks'][index_s]['lang_percent'] = {'UNKNOWN':1}

    fileDump = open(f'{artist}_spotify_lang_percent.json','w')
    json.dump(spotify_data,fileDump,indent=4)

In [ ]:
# files then moved to: processed_data/spotify_lang_percent
artists = ['bts','blackpink','exo','twice']
for artist in artists:
    tracksLangPercent(artist)